<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day24_Carnegie_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [1]:
url = ("https://raw.githubusercontent.com/ryanaxiom/Applied-ML/main/data/carnegie_data.csv")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score
carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
d["med_yn"] = (d["medical"] == 1).astype(int)
y = d["research"]
kf = KFold(n_splits=5, shuffle=True, random_state=309)
five = ["fallenr20", "totdeg", "facnum", "stem_rsd", "med_yn"]
print(carnegie.shape, d.shape)

(3939, 102) (302, 103)


In [ ]:
nh = pd.read_csv(url.replace("carnegie_data", "nhanes_adults"))
yd = (nh["Diabetes"] == "Yes").astype(int)
nh["male"] = (nh["Sex"] == "male").astype(int)
exam = ["Age", "BMI", "BPSysAve", "BPDiaAve", "Pulse",
        "TotChol", "DirectChol", "male"]
tr, te = next(iter(kf.split(nh)))                  # one fold
filled = nh[exam].fillna(nh[exam].iloc[tr].median())
from sklearn.ensemble import RandomForestClassifier
print(nh.shape, len(tr), len(te))

In [ ]:
from sklearn.inspection import permutation_importance
noisy = filled.copy()
noisy["noise"] = np.random.default_rng(309).uniform(size=len(noisy))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                            random_state=309)
rf.fit(noisy.iloc[tr], yd.iloc[tr])
built_in = pd.Series(rf.feature_importances_, index=noisy.columns)
print(built_in[["Age", "BMI", "noise", "male"]].round(3).to_dict())
shuffled = permutation_importance(rf, noisy.iloc[te], yd.iloc[te],
                                  scoring="roc_auc", n_repeats=5,
                                  random_state=309)
drop = pd.Series(shuffled.importances_mean, index=noisy.columns)
print(drop[["Age", "BMI", "noise", "male"]].round(3).to_dict())

In [ ]:
print(built_in.sort_values(ascending=False).round(3).to_dict())
print(drop.sort_values(ascending=False).round(3).to_dict())

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
x1 = d[["fallenr20"]]
stumps = GradientBoostingRegressor(max_depth=1, learning_rate=0.1, n_estimators=100,
                                   random_state=309)
stumps.fit(x1, y)
grid = x1.sort_values("fallenr20")                     # sort x first
plt.scatter(x1, y, s=8, alpha=0.4)
for k, pred in enumerate(stumps.staged_predict(grid), start=1):
    if k in [1, 10, 100]:
        plt.plot(grid, pred, label=f"{k} rounds")
        print(k, "rounds:", len(np.unique(pred.round(0))), "distinct fitted values")
plt.xlabel("fall enrollment 2020"); plt.ylabel("research ($ thousands)"); plt.legend(); plt.show()

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
boost = GradientBoostingClassifier(max_depth=3, learning_rate=0.1,
                                   n_estimators=600,
                                   random_state=309)
boost.fit(filled.iloc[tr], yd.iloc[tr])
fit = [log_loss(yd.iloc[tr], p[:, 1])
       for p in boost.staged_predict_proba(filled.iloc[tr])]
held = [log_loss(yd.iloc[te], p[:, 1])
        for p in boost.staged_predict_proba(filled.iloc[te])]
for k in [1, 10, 50, 100, 600]:
    print(k, round(fit[k - 1], 4), round(held[k - 1], 4))
print(int(np.argmin(held)) + 1)

In [ ]:
plt.plot(range(1, 601), fit, label="training rows")
plt.plot(range(1, 601), held, label="held-out rows")
plt.xlabel("rounds"); plt.ylabel("log loss"); plt.legend(); plt.show()
slow = GradientBoostingClassifier(max_depth=3, learning_rate=0.05, n_estimators=600,
                                  random_state=309)
slow.fit(filled.iloc[tr], yd.iloc[tr])
held_slow = [log_loss(yd.iloc[te], p[:, 1])
             for p in slow.staged_predict_proba(filled.iloc[te])]
print(int(np.argmin(held_slow)) + 1, round(min(held_slow), 4))

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
for trees in [10, 200, 1000]:
    rf_b = RandomForestClassifier(n_estimators=trees, min_samples_leaf=20,
                                  random_state=309, n_jobs=-1)
    forest = make_pipeline(SimpleImputer(strategy="median"), rf_b)
    ll = cross_val_score(forest, nh[exam], yd, cv=kf, scoring="neg_log_loss")
    print(trees, round(-ll.mean(), 4))

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
boosting = HistGradientBoostingClassifier(learning_rate=0.05,
                                          random_state=309)
auc = cross_val_score(boosting, nh[exam], yd, cv=kf,
                      scoring="roc_auc")
ll = cross_val_score(boosting, nh[exam], yd, cv=kf,
                     scoring="neg_log_loss")
print(round(auc.mean(), 4), round(-ll.mean(), 4))